In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Data Loading for confirmed, deaths and recovered cases into python using pandas

In [ ]:
try:
    confirmed = pd.read_excel("../dataset/covid_19_dataset.xlsx", sheet_name="covid_19_confirmed_v1")
    deaths = pd.read_excel("../dataset/covid_19_dataset.xlsx", sheet_name="covid_19_deaths_v1")
    recovered = pd.read_excel("../dataset/covid_19_dataset.xlsx", sheet_name="covid_19_recovered_v1")
    print("All datasets loaded successfully!")
except FileNotFoundError:
    print("Excel file not found. Please check the file path.")
except ValueError as e:
    print("Error reading Excel sheet:", e)

### 2. Data Exploration

**2.1.i. Structure of confirmed dataset in terms of rows, columns and datatypes**

In [ ]:
print(type(confirmed), '\n')
print("Rows and Columns:", confirmed.shape, '\n')
print(confirmed.dtypes)

**2.1.ii. Structure of deaths dataset in terms of rows, columns and datatypes**

In [ ]:
print(type(deaths), '\n')
print("Rows and Columns:", deaths.shape, '\n')
print(deaths.dtypes)

**2.1.iii. Structure of recovered dataset in terms of rows, columns and datatypes**

In [ ]:
print(type(recovered), '\n')
print("Rows and Columns:", recovered.shape, '\n')
print(recovered.dtypes)

### 2.2 Generate plots of confirmed cases over time for top countries

In [ ]:
# Identify date columns by excluding geographical columns
date_columns = [
    col for col in confirmed.columns
    if col not in ["Province/State", "Country/Region", "Lat", "Long"]
]

# print("date_columns", date_columns , "\n")
# Generate correct dates in their original column order
dates = pd.date_range(
    start="2020-01-22",
    periods=len(date_columns),
    freq="D"
)
# print("dates", dates , "\n")

# Identify the latest date column
latest_date_column = date_columns[-1]
# print("latest_date_column", latest_date_column , "\n")

# Calculate latest confirmed cases country-wise
country_totals = confirmed.groupby("Country/Region")[latest_date_column].sum().sort_values(ascending=False)

# print(country_totals)
# Select top 5 countries
top_countries = country_totals.head(5)

print("Top 5 Countries based on latest confirmed cases:")
print(top_countries)

In [ ]:
# Plot confirmed cases over time
# Filter top 5 countries and aggregate their data

top_country_data = confirmed[
    confirmed["Country/Region"].isin(top_countries.index)
].groupby("Country/Region")[date_columns].sum()

# print(top_country_data, "\n\n\n")

plt.figure(figsize = (15,7))

for country in top_country_data.index:
    # print(top_country_data.loc[country].values)
    plt.plot(dates, top_country_data.loc[country].values, label=country)

plt.title("COVID-19 Confirmed Cases Over Time - Top 5 Countries")
plt.xlabel("Date")
plt.ylabel("Cumulative Confirmed Cases")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()    
plt.show()

### 2.3 Generate plots of confirmed cases over time for China

In [ ]:
china_data = confirmed[confirmed["Country/Region"] == "China"]
# print(china_data.shape)
# display(china_data.head())

china_cases = china_data[date_columns].sum()
# print(china_cases)

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(dates, china_cases.values, label="china", color="red")
plt.title("COVID-19 Confirmed Cases Over Time - China")
plt.xlabel("Date")
plt.ylabel("Cumulative Confirmed Cases")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## 3. Handling Missing Data
Identify missing values and replace them using a suitable imputation method such as forward filling for time-series data.

In [ ]:
print("Confirmed Missing Values:")
print(confirmed.isnull().sum().sum())

print("\nDeaths Missing Values:")
print(deaths.isnull().sum().sum())

print("\nRecovered Missing Values:")
print(recovered.isnull().sum().sum())

In [ ]:
print("Confirmed:")
print(confirmed.isna().sum()[confirmed.isna().sum() > 0])

print("\nDeaths:")
print(deaths.isna().sum()[deaths.isna().sum() > 0])

print("\nRecovered:")
print(recovered.isna().sum()[recovered.isna().sum() > 0])

### 3.1 Identify rows containing missing values

In [ ]:
print("Confirmed - Missing Coordinates:")
display(confirmed[confirmed["Lat"].isnull() | confirmed["Long"].isnull() ])

print("\n\nDeaths - Missing Coordinates:")
display(deaths[deaths["Lat"].isna() | deaths["Long"].isna()])

print("\n\nRecovered - Missing Coordinates:")
display(recovered[recovered["Lat"].isnull() | recovered["Long"].isnull()])

In [ ]:
print("Confirmed - Missing Date Values:")
display(confirmed[confirmed["4/20/20"].isnull()])

print("\nDeaths - Missing Date Values:")
display(deaths[deaths["4/20/20"].isnull()])

print("\nRecovered - Missing Date Values:")
display(recovered[recovered["4/20/20"].isnull()])

### 3.2 Handle missing date values using Forward Fill

In [ ]:
# Forward fill missing values across date columns
confirmed[date_columns] = confirmed[date_columns].ffill(axis = 1)
deaths[date_columns] = deaths[date_columns].ffill(axis = 1)
recovered[date_columns] = recovered[date_columns].ffill(axis = 1)

In [ ]:
print("Confirmed - Missing Date Values:")
print(confirmed[date_columns].isnull().sum().sum())

print("\nDeaths - Missing Date Values:")
print(deaths[date_columns].isnull().sum().sum())

print("\nRecovered - Missing Date Values:")
print(recovered[date_columns].isnull().sum().sum())

In [ ]:
# Verify the two affected records
print("Deaths - Algeria:")
display(deaths.loc[deaths["Country/Region"] == "Algeria", ["Country/Region", "4/19/20", "4/20/20", "4/21/20"]])

print("\nRecovered - Argentina:")
display(recovered.loc[recovered["Country/Region"] == "Argentina", ["Country/Region",  "4/19/20", "4/20/20", "4/21/20"]])

## 4. Data Cleaning and Preparation
Replace blank values in Province/State column with "All Provinces".

In [ ]:
confirmed["Province/State"] = confirmed["Province/State"].fillna("All Provinces")
deaths["Province/State"] = deaths["Province/State"].fillna("All Provinces")
recovered["Province/State"] = recovered["Province/State"].fillna("All Provinces")

In [ ]:
print("Confirmed missing Province/State:",  confirmed["Province/State"].isnull().sum())
print("Deaths missing Province/State:", deaths["Province/State"].isnull().sum())
print("Recovered missing Province/State:", recovered["Province/State"].isnull().sum())

In [ ]:
# Check replacement values
print("Confirmed:")
print(confirmed["Province/State"].value_counts().head())

print("\nDeaths:")
print(deaths["Province/State"].value_counts().head())

print("\nRecovered:")
print(recovered["Province/State"].value_counts().head())

## 5. Independent Dataset Analysis

### 5.1 Analyze peak daily new cases in Germany, France, and Italy

In [ ]:
countries = ["Germany", "France", "Italy"]
country_data = (
    confirmed[confirmed["Country/Region"].isin(countries)]
).groupby("Country/Region")[date_columns].sum()

display(country_data)

In [ ]:
daily_new_cases = country_data.diff(axis = 1)
daily_new_cases

In [ ]:
daily_new_cases.iloc[:, 0] = country_data.iloc[:, 0]

display(daily_new_cases)

In [ ]:
# Identify peak daily cases
peak_cases = daily_new_cases.max(axis=1)
peak_dates = daily_new_cases.idxmax(axis=1)

peak_analysis = pd.DataFrame({
    "Peak Daily New Cases": peak_cases,
    "Peak Date": peak_dates
})

display(peak_analysis)

In [ ]:
# Identify the country with the highest surge
highest_surge_country = peak_analysis["Peak Daily New Cases"].idxmax()

print("Country with highest single-day surge:", highest_surge_country)

print("Highest daily new cases:", peak_analysis.loc[highest_surge_country, "Peak Daily New Cases"])

print("Date of highest surge:", peak_analysis.loc[highest_surge_country, "Peak Date"])

In [ ]:
france_peak_position = daily_new_cases.loc["France"].to_numpy().argmax()

print("france_peak_position", france_peak_position)
print("Peak daily cases:", daily_new_cases.loc["France"].max())
print("Correct peak date:", dates[france_peak_position])

In [ ]:
print("First date:", dates[0])
print("Last date:", dates[-1])
print("Total dates:", len(dates))

### 5.2 Recovery Rate Comparison: Canada vs Australia

Calculate the recovery rate for Canada and Australia as of December 31, 2020.

Recovery Rate = (Total Recovered Cases / Total Confirmed Cases) × 100

In [ ]:
# Identify the December 31, 2020 column
target_date = pd.Timestamp("2020-12-31")

date_position = dates.get_loc(target_date)

date_column = date_columns[date_position]

print("date_position", date_position)
print("Selected date:", dates[date_position])
print("Original dataset column:", date_column)

In [ ]:
# Calculate confirmed and recovered cases
countries = ["Canada", "Australia"]

confirmed_cases = (
    confirmed[confirmed["Country/Region"].isin(countries)]
    .groupby("Country/Region")[date_column]
    .sum()
)

recovered_cases = (
    recovered[recovered["Country/Region"].isin(countries)]
    .groupby("Country/Region")[date_column]
    .sum()
)

print("Confirmed Cases:")
print(confirmed_cases)

print("\nRecovered Cases:")
print(recovered_cases)

In [ ]:
# Calculate recovery rates
recovery_rate = (recovered_cases / confirmed_cases) * 100

recovery_comparison = pd.DataFrame({
    "Confirmed Cases": confirmed_cases,
    "Recovered Cases": recovered_cases,
    "Recovery Rate (%)": recovery_rate
})

display(recovery_comparison.round(2))

In [ ]:
# Identify the country with the higher recovery rate
better_country = recovery_rate.idxmax()

print("Country with higher recovery rate:", better_country)
print("Recovery rate:", round(recovery_rate.max(), 2), "%")

### 5.3 Death Rate Distribution Among Canadian Provinces

Calculate the death rate for each Canadian province using the latest available data.

Death Rate = (Total Deaths / Total Confirmed Cases) × 100

Identify the provinces with the highest and lowest death rates.

In [ ]:
# Filter Canadian data
canada_confirmed = confirmed[confirmed["Country/Region"] == "Canada"].copy()

canada_deaths = deaths[deaths["Country/Region"] == "Canada"].copy()

print("Confirmed dataset shape:", canada_confirmed.shape)
print("Deaths dataset shape:", canada_deaths.shape)

print("\nProvince/State names:")
print(sorted(canada_confirmed["Province/State"].unique()))

In [ ]:
# For this, we'll calculate the confirmed and death rates for the 13 provinces and territories only.
# We will remove 2 cruise-ship records: Diamond Princess and Grand Princess and also special record: Repatriated Travellers

canada_confirmed_filtered = canada_confirmed[~canada_confirmed["Province/State"].str.contains("Repatriated|Princess", case=False, na=False, regex=True)]
canada_deaths_filtered = canada_deaths[~canada_deaths["Province/State"].str.contains("Repatriated|Princess", case=False, na=False, regex=True)]

In [ ]:
print("Confirmed dataset shape:", canada_confirmed_filtered.shape)
print("Deaths dataset shape:", canada_deaths_filtered.shape)

print("\nProvince names:")
print(sorted(canada_confirmed_filtered["Province/State"].unique()))

In [ ]:
# Identify the latest date

latest_date_column = date_columns[-1]

print("Latest date:", dates[-1])
print("Dataset column:", latest_date_column)

In [ ]:
# Calculate total confirmed cases and deaths province-wise

province_confirmed = (canada_confirmed_filtered.groupby("Province/State")[latest_date_column].sum())

province_deaths = (canada_deaths_filtered.groupby("Province/State")[latest_date_column].sum())

print("Confirmed Cases:")
print(province_confirmed)

print("\nDeaths:")
print(province_deaths)

In [ ]:
# Calculate province-wise death rates

death_rate = (province_deaths / province_confirmed) * 100

canada_death_analysis = pd.DataFrame({
    "Confirmed Cases": province_confirmed,
    "Deaths": province_deaths,
    "Death Rate (%)": death_rate
})

canada_death_analysis = canada_death_analysis.sort_values(by="Death Rate (%)",ascending=False)

display(canada_death_analysis.round(2))


In [ ]:
# Identify highest and lowest death rates

highest_death_rate = canada_death_analysis["Death Rate (%)"].max()
lowest_death_rate = canada_death_analysis["Death Rate (%)"].min()

print("Highest death rate:")
display(canada_death_analysis[canada_death_analysis["Death Rate (%)"] == highest_death_rate])

print("\nLowest death rate:")
display(canada_death_analysis[canada_death_analysis["Death Rate (%)"] == lowest_death_rate])

### 6.1 Transform Deaths Dataset from Wide to Long Format

Convert the deaths dataset from wide format to long format using Pandas melt().

The transformed dataset should contain:
- Province/State
- Country/Region
- Lat
- Long
- Date
- Deaths

In [ ]:
deaths_for_melt = deaths.rename(columns=dict(zip(date_columns, dates)))

print(deaths_for_melt.shape)
print(deaths_for_melt.columns[:8])
deaths_for_melt.head()

In [ ]:
# Convert wide format to long format

deaths_long = deaths_for_melt.melt(id_vars=["Province/State", "Country/Region", "Lat", "Long"], var_name="Date", value_name="Deaths")

deaths_long["Date"] = pd.to_datetime(deaths_long["Date"])

display(deaths_long.head())

In [ ]:
print("Original deaths dataset shape:", deaths.shape)
print("Transformed deaths dataset shape:", deaths_long.shape)

print("\nColumn names:")
print(deaths_long.columns)

print("\nDate data type:")
print(deaths_long["Date"].dtype)

print("\nDate range:")
print(deaths_long["Date"].min())
print(deaths_long["Date"].max())

### 6.2 Total Deaths Reported Per Country

Calculate the total reported deaths for each country using the transformed long-format deaths dataset.

In [ ]:
# Group deaths by country

# Filter the latest date
latest_date = deaths_long["Date"].max()
latest_deaths = deaths_long[
    deaths_long["Date"] == latest_date
]

country_deaths = (latest_deaths.groupby("Country/Region")["Deaths"].sum().sort_values(ascending=False))
print(country_deaths.head())
print("Total deaths across all countries:", latest_deaths["Deaths"].sum())

### 6.3 Top 5 Countries with Highest Average Daily Deaths

Calculate daily new deaths from cumulative death counts and identify the five countries with the highest average daily deaths.

In [ ]:
# Calculate daily new deaths

daily_deaths = deaths_long.sort_values(by=["Country/Region", "Province/State", "Date"]).copy()

daily_deaths["Daily New Deaths"] = (daily_deaths.groupby(["Country/Region", "Province/State"])["Deaths"].diff())

daily_deaths["Daily New Deaths"] = daily_deaths["Daily New Deaths"].fillna(daily_deaths["Deaths"])

display(daily_deaths)

In [ ]:
# Calculate average daily deaths per country

average_daily_deaths = (daily_deaths.groupby("Country/Region")["Daily New Deaths"].mean().sort_values(ascending=False))

# Display the result
top_5_average_deaths = average_daily_deaths.head(5)

print("Top 5 countries with highest average daily deaths:")
print(top_5_average_deaths.round(2))

### 6.4 Death Trends in the United States

Analyze the cumulative deaths in the United States over time and visualize the trend using a line chart.

In [ ]:
# Filter US records

us_deaths = deaths_long[deaths_long["Country/Region"] == "US"].copy()

print("US dataset shape:", us_deaths.shape)
display(us_deaths.head())

In [ ]:
# Calculate total deaths by date

us_daily_deaths = (us_deaths.groupby("Date")["Deaths"].sum().sort_index())

print(us_daily_deaths)
print("\nLatest cumulative deaths:", us_daily_deaths.iloc[-1])

In [ ]:
# Plot the trend

plt.figure(figsize=(15, 7))

plt.plot(us_daily_deaths.index, us_daily_deaths.values, label="United States", color="red")

plt.title("Cumulative COVID-19 Deaths in the United States")
plt.xlabel("Date")
plt.ylabel("Total Deaths")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

**Conclusion:**

The cumulative COVID-19 deaths in the United States increased significantly between January 2020 and May 2021.

The steepest increase occurred approximately between December 2020 and February 2021. By May 29, 2021, the United States had recorded 594,306 cumulative deaths.

The line chart illustrates the progression of total reported deaths over the study period.

### 7.1 Merge Confirmed, Deaths and Recovered Datasets

Transform the confirmed and recovered datasets into long format and merge them with the transformed deaths dataset using Country/Region and Date.

In [ ]:
# Transform confirmed cases into long format

confirmed_for_melt = confirmed.rename(columns=dict(zip(date_columns, dates)))

confirmed_long = confirmed_for_melt.melt(id_vars=["Province/State", "Country/Region", "Lat", "Long"], var_name="Date", value_name="Confirmed")

confirmed_long["Date"] = pd.to_datetime(confirmed_long["Date"])

display(confirmed_long.head())
print("Shape:", confirmed_long.shape)

In [ ]:
# Transform recovered cases into long format

recovered_for_melt = recovered.rename(columns=dict(zip(date_columns, dates)))

recovered_long = recovered_for_melt.melt(id_vars=["Province/State", "Country/Region", "Lat", "Long"], var_name="Date", value_name="Recovered")

recovered_long["Date"] = pd.to_datetime(recovered_long["Date"])

display(recovered_long.head())
print("Shape:", recovered_long.shape)

In [ ]:
# Verify all three transformed datasets

print("Confirmed:", confirmed_long.shape)
print("Deaths:", deaths_long.shape)
print("Recovered:", recovered_long.shape)

print("\nConfirmed date type:", confirmed_long["Date"].dtypes)
print("Deaths date type:", deaths_long["Date"].dtype)
print("Recovered date type:", recovered_long["Date"].dtype)

In [ ]:
# Aggregate confirmed cases by country and date

confirmed_country = (confirmed_long.groupby(["Country/Region", "Date"])["Confirmed"].sum().reset_index())

display(confirmed_country.head())
print("Shape:", confirmed_country.shape)

In [ ]:
# Aggregate deaths by country and date

deaths_country = (deaths_long.groupby(["Country/Region", "Date"])["Deaths"].sum().reset_index())

display(deaths_country.head())
print("Shape:", deaths_country.shape)

In [ ]:
# Aggregate recovered cases by country and date

recovered_country = (recovered_long.groupby(["Country/Region", "Date"])["Recovered"].sum().reset_index())

display(recovered_country.head())
print("Shape:", recovered_country.shape)

In [ ]:
# Merge all three datasets

merged_data = pd.merge(confirmed_country, deaths_country, on=["Country/Region", "Date"], how="outer")

merged_data = pd.merge(merged_data, recovered_country, on=["Country/Region", "Date"], how="outer")

display(merged_data.head())
print("Merged dataset shape:", merged_data.shape)

In [ ]:
print("Dataset information:")
print(merged_data.info())

print("\nMissing values:")
print(merged_data.isnull().sum())

print("\nDuplicate country-date combinations:",
      merged_data.duplicated(
          subset=["Country/Region", "Date"]
      ).sum())

### 7.2 Monthly Analysis of Confirmed Cases, Deaths and Recoveries

Analyze monthly totals of confirmed cases, deaths and recoveries for each country using the merged dataset.

In [ ]:
# Extract year and month

merged_data["Year-Month"] = merged_data["Date"].dt.to_period("M")

display(merged_data)

In [ ]:
# Calculate monthly totals

monthly_data = (
    merged_data
    .groupby(["Country/Region", "Year-Month"])[
        ["Confirmed", "Deaths", "Recovered"]
    ]
    .sum()
    .reset_index()
)

display(monthly_data.head(10))

In [ ]:
print("Monthly dataset shape:", monthly_data.shape)

print("\nMissing values:")
print(monthly_data.isnull().sum())

print("\nAvailable months:")
print(monthly_data["Year-Month"].nunique())

### 7.3 Monthly Analysis for US, Italy and Brazil

Analyze the monthly sums of confirmed cases, deaths and recoveries for the United States, Italy and Brazil using the merged dataset.

In [ ]:
# Filter the three countries

selected_countries = ["US", "Italy", "Brazil"]

selected_monthly_data = monthly_data[monthly_data["Country/Region"].isin(selected_countries)].copy()

display(selected_monthly_data.head(10))

In [ ]:
print("Dataset shape:", selected_monthly_data.shape)

print("\nCountries:")
print(selected_monthly_data["Country/Region"].unique())

print("\nMonthly records per country:")
print(selected_monthly_data.groupby("Country/Region").size())

In [ ]:
# Visualize monthly confirmed cases

plt.figure(figsize=(15, 7))

for country in selected_countries:
    country_data = selected_monthly_data[selected_monthly_data["Country/Region"] == country]

    plt.plot(country_data["Year-Month"].astype(str), country_data["Confirmed"], marker="o", label=country)

plt.title("Monthly Sum of Confirmed COVID-19 Cases")
plt.xlabel("Year-Month")
plt.ylabel("Monthly Sum of Confirmed Cases")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Monthly deaths comparison


plt.figure(figsize=(15, 7))

for country in selected_countries:
    country_data = selected_monthly_data[selected_monthly_data["Country/Region"] == country]

    plt.plot(country_data["Year-Month"].astype(str), country_data["Deaths"], marker="o", label=country)

plt.title("Monthly Sum of COVID-19 Deaths")
plt.xlabel("Year-Month")
plt.ylabel("Monthly Sum of Deaths")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Monthly recoveries comparison

plt.figure(figsize=(15, 7))

for country in selected_countries:
    country_data = selected_monthly_data[selected_monthly_data["Country/Region"] == country]

    plt.plot(country_data["Year-Month"].astype(str), country_data["Recovered"], marker="o", label=country)

plt.title("Monthly Sum of COVID-19 Recoveries")
plt.xlabel("Year-Month")
plt.ylabel("Monthly Sum of Recoveries")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Compare the three countries

latest_month = selected_monthly_data["Year-Month"].max()

latest_comparison = selected_monthly_data[selected_monthly_data["Year-Month"] == latest_month]

display(latest_comparison)

**Data Observation:**

The US recovery dataset contains zero values from January 2021 onwards. Therefore, the recovery trend for the United States may not accurately represent actual recoveries during this period. This limitation should be considered when interpreting the comparison.

### 8.1 Top 3 Countries with Highest Average Death Rates in 2020

Calculate the daily death rate for each country using the combined dataset and identify the three countries with the highest average death rates throughout 2020.

Death Rate = (Deaths / Confirmed Cases) × 100

In [ ]:
# Filter data for 2020

data_2020 = merged_data[merged_data["Date"].dt.year == 2020].copy()

display(data_2020.head())
print("\nDataset shape:", data_2020.shape)

In [ ]:
# Calculate daily death rates

data_2020["Death Rate (%)"] = (data_2020["Deaths"] / data_2020["Confirmed"]) * 100
data_2020

In [ ]:
data_2020.replace(
    [float("inf"), float("-inf")],
    float("nan"),
    inplace=True
)

data_2020 = data_2020.dropna(subset=["Death Rate (%)"])
data_2020

In [ ]:
# Calculate average death rate country-wise

average_death_rate = (data_2020.groupby("Country/Region")["Death Rate (%)"].mean().sort_values(ascending=False))

print("Top 3 countries with highest average death rates:")
print(average_death_rate.head(3).round(2))

In [ ]:
top_3_death_rates = average_death_rate.head(3)

print(top_3_death_rates)

In [ ]:
# Exclude cruise-ship records

excluded_entities = [
    "MS Zaandam",
    "Diamond Princess",
    "Grand Princess"
]

data_2020_countries = data_2020[~data_2020["Country/Region"].isin(excluded_entities)].copy()

In [ ]:
# Recalculate the average death rates

average_death_rate = (data_2020_countries.groupby("Country/Region")["Death Rate (%)"].mean().sort_values(ascending=False))

top_3_death_rates = average_death_rate.head(3)

print("Top 3 countries with highest average death rates:")
print(top_3_death_rates.round(2))

**Conclusion:**

Yemen recorded the highest average COVID-19 death rate during 2020 at 24.95%, followed by Sudan (10.37%) and Italy (9.72%).

These higher rates indicate a greater proportion of reported deaths relative to confirmed cases. Differences in healthcare infrastructure, testing availability and reporting practices may have influenced these outcomes.

### 8.2 Compare Total Recoveries and Deaths in South Africa

Calculate the total reported recoveries and deaths in South Africa using the latest available date in the merged dataset.

In [ ]:
# Filter South Africa

south_africa = merged_data[merged_data["Country/Region"] == "South Africa"].copy()

print(south_africa.head())
print("Dataset shape:", south_africa.shape)

In [ ]:
# Identify the latest date

latest_date = south_africa["Date"].max()

south_africa_latest = south_africa[south_africa["Date"] == latest_date]

print("Latest date:", latest_date)
print(south_africa_latest)

In [ ]:
# Compare recoveries and deaths

total_recoveries = south_africa_latest["Recovered"].sum()
total_deaths = south_africa_latest["Deaths"].sum()

print("Total recoveries:", total_recoveries)
print("Total deaths:", total_deaths)

print("Recoveries-to-deaths ratio:", round(total_recoveries / total_deaths, 2))

**Conclusion:**

As of May 29, 2021, South Africa reported 1,554,184 recoveries and 56,363 deaths.

The recoveries-to-deaths ratio was approximately 27.57:1, meaning that for every reported death, approximately 27.57 recoveries were recorded.

This indicates that reported recoveries considerably outnumbered reported deaths in South Africa. However, these figures do not represent all case outcomes, as some cases may still have been active.

### 8.3 Monthly Recovery Ratio for the United States

Calculate the monthly recovery ratio for the United States from March 2020 to May 2021.

Recovery Ratio = (Recoveries / Confirmed Cases) × 100

In [ ]:
# Filter US data

us_monthly_data = monthly_data[monthly_data["Country/Region"] == "US"].copy()

us_monthly_data = us_monthly_data[(us_monthly_data["Year-Month"] >= "2020-03") & (us_monthly_data["Year-Month"] <= "2021-05")].copy()

print(us_monthly_data)

In [ ]:
# Calculate monthly recovery ratio

us_monthly_data["Recovery Ratio (%)"] = (us_monthly_data["Recovered"] / us_monthly_data["Confirmed"]) * 100

print(us_monthly_data[["Year-Month", "Confirmed", "Recovered", "Recovery Ratio (%)"]].round(2))

In [ ]:
# Find the month with the highest recovery ratio

highest_recovery_month = us_monthly_data.loc[us_monthly_data["Recovery Ratio (%)"].idxmax()]

print("Month with highest recovery ratio:", highest_recovery_month["Year-Month"])

print("Highest recovery ratio:", round(highest_recovery_month["Recovery Ratio (%)"], 2), "%")

In [ ]:
# Visualize the monthly recovery ratio

plt.figure(figsize=(15, 7))

plt.plot(us_monthly_data["Year-Month"].astype(str), us_monthly_data["Recovery Ratio (%)"], marker="o", label="US Recovery Ratio")

plt.title("Monthly COVID-19 Recovery Ratio - United States")
plt.xlabel("Year-Month")
plt.ylabel("Recovery Ratio (%)")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

**Conclusion:**

The United States recorded its highest monthly recovery ratio in October 2020 at 39.58%.

The recovery ratio increased steadily from March 2020 to October 2020 and then declined towards December 2020.

The recovery ratio became zero from January 2021 onwards because the source dataset contains zero recovery values for the United States during this period.

Possible reasons for changes in recovery ratios include improvements in recovery reporting, changes in the number of confirmed cases, differences in recovery reporting practices, and delays between diagnosis and recovery.

Therefore, the zero recovery ratios in 2021 should be interpreted as a data limitation rather than evidence that no patients recovered.